# openpkflow â€” Full Feature Tour

A transparent, reproducible, open-source Python workflow for dissolution, NCA, PK/PD simulation, and pharmacometric reporting.

**Modules covered**
1. Package info
2. Dissolution â€” f1/f2, MSD, bootstrap, model fitting, multi-media
3. NCA â€” core functions, full study, steady-state, urinary excretion, sparse sampling
4. PK Simulation â€” 1-cmt and 2-cmt, IV/oral, repeated dosing
5. IVIVC â€” Wagner-Nelson, Loo-Riegelman, convolution, Levy plot, predictability
6. Population PK â€” GOF metrics, IWRES, VPC
7. Bayesian PK â€” MAP estimation (no extra deps), full posterior (requires PyMC)
8. Bioequivalence â€” TOST, BEStudy
9. ML Surrogate â€” PKSurrogate (requires PyTorch)
10. Report generation â€” HTML, PDF, DOCX
11. CLI quick reference

> **Disclaimer:** This notebook was generated using OpenPKFlow (open-source). Final regulatory interpretation should be reviewed by qualified formulation, pharmacokinetic, and regulatory experts.

In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import openpkflow

print("openpkflow version:", openpkflow.__version__)
print("Author:", openpkflow.__author__)

openpkflow version: 2.0.0
Author: Priyam Thakar


---
## 1. Dissolution â€” Similarity Metrics (f1, f2, MSD)

In [11]:
from openpkflow.dissolution import f1, f2, max_deviation, msd

# Dissolution profiles (% released at each time point)
reference = [14.5, 30.0, 47.5, 61.7, 78.0, 90.0]
test_similar = [13.0, 28.5, 45.0, 59.0, 75.5, 87.8]  # within 10%
test_different = [10.0, 20.0, 35.0, 50.0, 65.0, 78.0]  # clearly different

print("=== Profiles that should pass ===")
print(f"f1 = {f1(reference, test_similar):.2f}  (pass: <= 15)")
print(f"f2 = {f2(reference, test_similar, method='all_points'):.2f}  (pass: >= 50)")
print(f"f2 regulatory = {f2(reference, test_similar, method='regulatory'):.2f}")

print("\n=== Profiles that should fail ===")
print(f"f1 = {f1(reference, test_different):.2f}")
print(f"f2 = {f2(reference, test_different, method='all_points'):.2f}")

print("\n=== Sanity check: identical profiles ===")
print(f"f1 identical = {f1(reference, reference):.2f}  (expected: 0)")
print(f"f2 identical = {f2(reference, reference, method='all_points'):.2f}  (expected: 100)")

print("\n=== Other metrics ===")
print(f"Max deviation = {max_deviation(reference, test_similar):.2f}%")

msd_result = msd(reference, test_similar)
print(f"MSD value = {msd_result.msd:.4f}")
print(f"Is similar (chi2 test) = {msd_result.is_similar}")
print(msd_result.summary())

=== Profiles that should pass ===
f1 = 4.01  (pass: <= 15)
f2 = 80.81  (pass: >= 50)
f2 regulatory = 80.81

=== Profiles that should fail ===
f1 = 19.80
f2 = 47.84

=== Sanity check: identical profiles ===
f1 identical = 0.00  (expected: 0)
f2 identical = 100.00  (expected: 100)

=== Other metrics ===
Max deviation = 2.70%
MSD value = 2.2361
Is similar (chi2 test) = True
Mahalanobis Statistical Distance (MSD)
Timepoints: 6
MSD squared: 5.0000
MSD: 2.2361
Chi2(0.05, 6): 12.5916
Verdict: SIMILAR



### 1.1 Bootstrap f2 Confidence Interval

In [12]:
from openpkflow.dissolution import bootstrap_f2

# bootstrap_f2 expects 2-D arrays: shape (n_vessels, n_timepoints)
rng_bs = np.random.default_rng(0)
ref_means = [14.5, 30.0, 47.5, 61.7, 78.0, 90.0]
tst_means = [13.0, 28.5, 45.0, 59.0, 75.5, 87.8]

# Simulate 6 vessels per formulation with small within-vessel noise
ref_2d = rng_bs.normal(loc=ref_means, scale=2.0, size=(6, 6)).clip(0, 100)
tst_2d = rng_bs.normal(loc=tst_means, scale=2.0, size=(6, 6)).clip(0, 100)

bs_result = bootstrap_f2(ref_2d, tst_2d, n_replicates=1000, confidence_level=0.90)
print(bs_result.summary())
print(f"f2 observed = {bs_result.f2_observed:.2f}")
print(f"90% CI: [{bs_result.ci_lower:.2f}, {bs_result.ci_upper:.2f}]")
print(f"Similar (lower CI >= 50): {bs_result.is_similar}")

TypeError: bootstrap_f2() got an unexpected keyword argument 'ci_percent'

### 1.2 DissolutionStudy â€” high-level workflow from CSV

In [ ]:
from pathlib import Path

# Find the bundled example dataset
import openpkflow.datasets as _ds
from openpkflow.dissolution import DissolutionStudy

datasets_dir = Path(_ds.__file__).parent
csv_path = datasets_dir / "example_dissolution.csv"

study = DissolutionStudy.from_csv(csv_path)
print("Formulations found:", study.formulations())

comparison = study.compare("reference", "test")
print(comparison.summary())

Formulations found: ['reference', 'test']
Dissolution Similarity Analysis
Reference: reference  |  Test: test
Timepoints: 6  |  Method: f1/f2 (FDA 1997 guidance)

f1 (difference factor): 12.34
f2 (similarity factor): 57.87

Interpretation: f2 >= 50 supports similarity between profiles.

Disclaimer: This output was generated using OpenPKFlow (open-source).
Final regulatory interpretation should be reviewed by qualified experts.


In [13]:
# Generate an HTML report
comparison.report("dissolution_comparison.html", format="html")
print("Report written: dissolution_comparison.html")

Report written: dissolution_comparison.html


### 1.3 Dissolution Model Fitting

In [14]:
from openpkflow.dissolution import fit_dissolution_models

time_points = [5, 10, 15, 20, 30, 45, 60, 90]
observed_pct = [8, 18, 32, 48, 68, 82, 91, 97]

fit_results = fit_dissolution_models(
    time_points=time_points,
    observed_mean=observed_pct,
    formulation_label="Prototype A",
)

print(fit_results.summary())

best = fit_results.best
print(f"\nBest model: {best.model_name}")
print(f"  R2={best.r_squared:.4f}, AIC={best.aic:.2f}, AICc={best.aicc:.2f}")
print(f"  Params: {best.params}")

# Predict at new time points
t_new = np.linspace(0, 120, 200)
predicted = best.predict(t_new)

plt.figure(figsize=(7, 4))
plt.scatter(time_points, observed_pct, color="black", zorder=5, label="Observed")
plt.plot(t_new, predicted, label=f"Best fit: {best.model_name}")
plt.xlabel("Time (min)")
plt.ylabel("% Released")
plt.title("Dissolution Model Fit")
plt.legend()
plt.tight_layout()
plt.show()

Dissolution Model Fitting
Formulation:  Prototype A
Timepoints:   8  |  Fit target: mean profile
Models fitted: 5  |  Converged: 5

Model                      R2      AICc       BIC  Params                                  Rank
--------------------------------------------------------------------------------------------
weibull                0.9950     19.37     17.13  alpha=1.364  beta=28.99                 1 [BEST]
first_order            0.9656     31.09     30.50  k1=0.03274                              2
higuchi                0.8926     40.19     39.60  kH=10.77                                3
korsmeyer_peppas       0.9132     42.22     39.98  k=7.306  n=0.6005                       4
zero_order             0.6962     48.51     47.92  k0=1.404                                5

Note: Ranked by AICc (lower is better). R2 shown for reference only.
Fit characterises release mechanism; it is not a regulatory similarity
test. Use f2 or bootstrap f2 for dissolution similarity assessment

C:\Users\priya\AppData\Local\Temp\ipykernel_6516\291597569.py:6: UserWarning: Korsmeyer-Peppas: 4 timepoints exceed 60% release. The power-law model is only mechanistically valid up to ~60% dissolved. Consider subsetting to early timepoints before fitting.
  fit_results = fit_dissolution_models(
C:\Users\priya\AppData\Local\Temp\ipykernel_6516\291597569.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Fit all available models and compare
from openpkflow.dissolution.models import VALID_MODELS

print("Available dissolution models:", VALID_MODELS)

# All individual fits
header = f"{'Model':<20} {'R2':>8} {'AIC':>10} {'AICc':>10} {'BIC':>10} Converged"
print()
print(header)
print("-" * 65)
for fit in fit_results.fits:
    row = (
        f"{fit.model_name:<20} {fit.r_squared:>8.4f}"
        f" {fit.aic:>10.2f} {fit.aicc:>10.2f}"
        f" {fit.bic:>10.2f} {fit.converged}"
    )
    print(row)

Available dissolution models: frozenset({'higuchi', 'first_order', 'zero_order', 'weibull', 'korsmeyer_peppas'})

Model                      R2        AIC       AICc        BIC Converged
-----------------------------------------------------------------
zero_order             0.6962      47.84      48.51      47.92 True
first_order            0.9656      30.42      31.09      30.50 True
higuchi                0.8926      39.52      40.19      39.60 True
korsmeyer_peppas       0.9132      39.82      42.22      39.98 True
weibull                0.9950      16.97      19.37      17.13 True


### 1.4 Model-Dependent Comparison

In [16]:
from openpkflow.dissolution import model_dependent_comparison

ref_time = [5, 10, 15, 20, 30, 45, 60]
ref_pct = [10, 22, 38, 54, 73, 87, 94]
tst_time = [5, 10, 15, 20, 30, 45, 60]
tst_pct = [9, 20, 35, 50, 70, 84, 92]

mc_result = model_dependent_comparison(
    ref_time_points=ref_time,
    ref_observed_mean=ref_pct,
    tst_time_points=tst_time,
    tst_observed_mean=tst_pct,
    model="weibull",
    param_index=0,  # compare first parameter (scale)
    ci_range=(80, 125),
)
print(mc_result.summary())

Model: weibull  Param: alpha  Ratio: 98.9%  90% CI: [87.5%, 110.3%]  Verdict: SIMILAR


D:\openpkflow\src\openpkflow\dissolution\models.py:46: RuntimeWarning: invalid value encountered in power
  100.0 * (1.0 - np.exp(-np.power(np.clip(t, 0.0, None) / beta, alpha))), dtype=float


### 1.5 Multi-Media Dissolution

In [17]:
import tempfile

from openpkflow.dissolution import MultiMediaStudy


# Build synthetic CSV data for three pH media
def make_csv(ref_vals, tst_vals, times):
    rows = []
    for t, r, ts in zip(times, ref_vals, tst_vals, strict=False):
        rows.append(f"reference,R1,{t},{r}")
        rows.append(f"test,T1,{t},{ts}")
    return "formulation,batch,time,percent_released\n" + "\n".join(rows)


times = [15, 30, 45, 60, 90]
media_data = {
    "pH 1.2": make_csv([20, 42, 65, 80, 92], [18, 40, 62, 78, 90], times),
    "pH 4.5": make_csv([22, 45, 68, 83, 94], [20, 43, 65, 80, 92], times),
    "pH 6.8": make_csv([25, 50, 72, 87, 96], [23, 47, 69, 85, 94], times),
}

# Write temp files
with tempfile.TemporaryDirectory() as tmpdir:
    media_csvs = {}
    for media_name, csv_content in media_data.items():
        p = Path(tmpdir) / f"{media_name.replace(' ', '_')}.csv"
        p.write_text(csv_content)
        media_csvs[media_name] = p

    mm_study = MultiMediaStudy(
        media_csvs=media_csvs,
        reference_label="reference",
        test_label="test",
    )
    mm_result = mm_study.run()

print(mm_result.summary())
print("f2 per media:", mm_result.f2_summary)
print("Overall pass:", mm_result.overall_pass)

Multi-Media Dissolution Comparison
Reference: reference
Test:      test
Media:     pH 1.2, pH 4.5, pH 6.8

Medium             f2  Status
------------ --------  --------
pH 1.2          80.55  PASS
pH 4.5          78.87  PASS
pH 6.8          78.87  PASS

Overall: PASS
f2 per media: {'pH 1.2': 80.54621874040892, 'pH 4.5': 78.87254899964358, 'pH 6.8': 78.87254899964358}
Overall pass: True


D:\openpkflow\src\openpkflow\dissolution\multi_media.py:226: UserWarning: More than one mean dissolution value exceeds 85% in the reference profile (2 timepoints above 85%). Per common regulatory practice, only one timepoint above 85% should be included in the f2 calculation. Review your timepoint selection.
  per_media[medium] = study.compare(self._reference_label, self._test_label)


---
## 2. NCA â€” Non-Compartmental Analysis

### 2.1 Core NCA Functions

In [18]:
from openpkflow.nca import (
    auc_inf_obs,
    auc_linear,
    auc_linear_up_log_down,
    auc_log,
    auc_percent_extrapolated,
    clearance_volume_parameters,
    cmax,
    lambda_z,
    tmax,
)

# Theophylline-like profile (Subject 1 from bundled dataset)
times = [0.00, 0.25, 0.57, 1.12, 2.02, 3.82, 5.10, 7.03, 9.05, 12.12, 24.37]
concs = [0.74, 2.84, 6.57, 10.50, 9.66, 8.58, 8.36, 7.47, 6.89, 5.94, 3.28]
dose = 320.0  # mg

# AUC
auclast_lin = auc_linear(times, concs)
auclast_log = auc_log(times, concs)
auclast_luld = auc_linear_up_log_down(times, concs)
print(f"AUClast (linear)       = {auclast_lin:.2f}")
print(f"AUClast (log)          = {auclast_log.value:.2f}")
print(f"AUClast (lin-up/log-dn)= {auclast_luld.value:.2f}")

# Basic parameters
print(f"\nCmax = {cmax(concs):.2f} mg/L")
print(f"Tmax = {tmax(times, concs):.2f} h")

# Terminal elimination
lz = lambda_z(times, concs, method="auto")
print(f"\nlambda_z = {lz.lambda_z:.4f} 1/h")
print(f"t1/2     = {lz.half_life:.2f} h")
print(f"R2       = {lz.r_squared:.4f}  (adj R2 = {lz.adj_r_squared:.4f})")
print(f"Points used: {lz.n_points} (from t={lz.time_start:.2f} to {lz.time_end:.2f})")

# AUCinf and extrapolation
aucinf = auc_inf_obs(auclast_luld.value, concs[-1], lz)
pct_ext = auc_percent_extrapolated(auclast_luld.value, aucinf)
print(f"\nAUCinf_obs  = {aucinf:.2f}")
print(f"% extrapolated = {pct_ext:.1f}%")

# Clearance & volume
params = clearance_volume_parameters(dose, aucinf, lz, route="oral")
print(f"\nCL_F = {params['CL_F']:.2f} L/h")
print(f"Vz_F = {params['Vz_F']:.2f} L")

AUClast (linear)       = 148.92
AUClast (log)          = 147.01
AUClast (lin-up/log-dn)= 147.23

Cmax = 10.50 mg/L
Tmax = 1.12 h

lambda_z = 0.0485 1/h
t1/2     = 14.30 h
R2       = 1.0000  (adj R2 = 1.0000)
Points used: 3 (from t=9.05 to 24.37)

AUCinf_obs  = 214.92
% extrapolated = 31.5%

CL_F = 1.49 L/h
Vz_F = 30.73 L


### 2.2 NCAStudy â€” full multi-subject analysis

In [19]:
from openpkflow.nca import NCAStudy

nca_csv = datasets_dir / "theoph.csv"

study = NCAStudy.from_csv(
    nca_csv,
    auc_method="linear_up_log_down",
    blq_method="drop",
)
results = study.analyze()

print(results.summary())

# Inspect as DataFrame
df = results.to_dataframe()
pk_cols = ["subject", "Cmax", "Tmax", "AUClast", "AUCinf_obs", "half_life", "CL_F", "Vz_F"]
print("\nKey columns:", [c for c in df.columns if c in pk_cols])
df[pk_cols].head(6)

ValueError: Unknown blq_method 'omit'. Valid values: ['drop', 'half_lloq', 'lloq', 'none', 'zero'] (aliases: m1->drop, m2->zero).

In [20]:
# Generate NCA HTML report
results.report("nca_summary.html", format="html")
print("NCA report written: nca_summary.html")

NameError: name 'results' is not defined

### 2.3 Steady-State NCA

In [21]:
from openpkflow.nca import accumulation_ratio, auc_tau, steady_state_parameters

# Simulate a steady-state dosing interval (tau = 12 h)
ss_times = [0, 0.5, 1, 2, 4, 6, 8, 12]
ss_concs = [4.5, 6.8, 9.2, 11.0, 10.2, 8.8, 7.2, 4.6]  # Cmin == C(tau)

tau = 12.0
auc_ss = auc_tau(ss_times, ss_concs, tau=tau, method="linear_up_log_down")
print(f"AUCtau (ss) = {auc_ss:.2f}")

ss_params = steady_state_parameters(ss_times, ss_concs, tau=tau, auc_method="linear_up_log_down")
for k, v in ss_params.items():
    if v is not None:
        print(f"  {k} = {v:.3f}")

# Accumulation ratio (needs single-dose AUCtau)
# Here we approximate single-dose AUCinf as AUCtau_sd for illustration
auctau_sd = 60.0  # hypothetical single-dose AUCtau
acc = accumulation_ratio(auc_ss, auctau_sd)
print(f"\nAccumulation ratio = {acc:.2f}x")

AUCtau (ss) = 96.24
  Cmax_ss = 11.000
  Cmin_ss = 4.500
  Cavg_ss = 8.020
  AUCtau = 96.240
  fluctuation_pct = 81.047
  swing = 1.444

Accumulation ratio = 1.60x


### 2.4 Urinary Excretion

In [22]:
from openpkflow.nca import cumulative_urinary_excretion, percent_excreted, renal_clearance

# Urine collection intervals (end times in hours)
urine_times = [2, 4, 8, 12, 24]
urine_volumes = [0.3, 0.4, 0.6, 0.5, 1.2]  # litres
urine_concs = [120, 95, 60, 40, 20]  # mg/L

Ae = cumulative_urinary_excretion(urine_times, urine_volumes, urine_concs)
print("Cumulative Ae (mg) at each time:", [f"{v:.1f}" for v in Ae])
print(f"Total Ae = {Ae[-1]:.1f} mg")

aucinf_val = 150.0  # from NCA
CLr = renal_clearance(total_ae=Ae[-1], auc_inf=aucinf_val)
print(f"Renal clearance (CLr) = {CLr:.2f} L/h")

pct = percent_excreted(total_ae=Ae[-1], dose=500.0)
print(f"% dose excreted = {pct:.1f}%")

Cumulative Ae (mg) at each time: ['36.0', '74.0', '110.0', '130.0', '154.0']
Total Ae = 154.0 mg
Renal clearance (CLr) = 1.03 L/h
% dose excreted = 30.8%


### 2.5 Sparse-Sampling NCA (model-informed)

In [23]:
from openpkflow.nca import fit_sparse_1cmt_oral, sparse_nca_bias_analysis

# Only 4 sparse samples from a real-world scenario
sparse_times = [0.5, 2.0, 6.0, 24.0]
sparse_concs = [1.8, 4.5, 3.8, 1.2]  # mg/L

sparse_result = fit_sparse_1cmt_oral(
    times=sparse_times,
    concentrations=sparse_concs,
    dose=200.0,
)

print(sparse_result.summary())
print(f"\nConverged: {sparse_result.converged}")
print(f"CL_F = {sparse_result.CL_F:.2f} L/h")
print(f"Vz_F = {sparse_result.Vz_F:.2f} L")
print(f"ka   = {sparse_result.ka:.3f} 1/h")
print(f"t1/2 = {sparse_result.half_life:.2f} h")
print(f"AUCinf = {sparse_result.AUCinf:.2f}")

# Plot fitted vs observed
sparse_result.plot(show=True)

Sparse NCA Results
Route: oral | Dose: 200 mg | Samples: 4
Converged: Yes

Fitted Parameters (1-cmt oral):
  CL_F  = 2.503 L/h
  Vz_F  = 35.84 L
  ka    = 0.9137 1/h
  k     = 0.06984 1/h
  t1/2  = 9.925 h

Standard Errors:
  CL_F  = 0.4239 L/h
  Vz_F  = 4.065 L
  ka    = 0.2354 1/h

Derived Parameters:
  AUClast  = 66.9 h*ng/mL
  AUCinf   = 79.91 h*ng/mL
  Cmax     = 4.511 ng/mL
  Tmax     = 3.03 h

Observed vs Fitted:
    Time          Obs          Fit        Resid
    0.50          1.8        2.009      -0.2086
    2.00          4.5        4.283       0.2169
    6.00          3.8        3.949       -0.149
   24.00          1.2        1.131      0.06937

Converged: True
CL_F = 2.50 L/h
Vz_F = 35.84 L
ka   = 0.914 1/h
t1/2 = 9.93 h
AUCinf = 79.91


D:\openpkflow\src\openpkflow\nca\sparse.py:295: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# Bias analysis: compare sparse-derived estimates to a hypothetical "rich" NCA result
# sparse_nca_bias_analysis accepts any object with the expected attribute names
from dataclasses import dataclass


@dataclass
class MockRichResult:
    AUClast: float = 22.0
    AUCinf_obs: float = 25.0  # attribute name must be AUCinf_obs
    Cmax: float = 5.2
    Tmax: float = 2.0
    half_life: float = 8.5
    CL_F: float = 8.0
    Vz_F: float = 42.0


bias = sparse_nca_bias_analysis(sparse_result, MockRichResult())
print("Bias analysis:")
for param, pct_bias in bias["biased_parameters"].items():
    if pct_bias is not None:
        print(f"  {param}: {pct_bias:+.1f}%")
    else:
        print(f"  {param}: N/A")

Bias analysis:
  AUClast_pct_bias: +204.1%
  AUCinf_pct_bias: +219.6%
  Cmax_pct_bias: -13.3%
  Tmax_pct_bias: +51.5%
  half_life_pct_bias: +16.8%
  CL_F_pct_bias: -68.7%
  Vz_F_pct_bias: -14.7%


---
## 3. PK Simulation

### 3.1 Analytical 1-compartment functions

In [25]:
from openpkflow.sim import (
    c_1cmt_iv_bolus,
    c_1cmt_iv_infusion,
    c_1cmt_oral,
    c_2cmt_iv_bolus,
    c_2cmt_oral,
    superpose,
)

t = np.linspace(0, 24, 200)

# 1-compartment routes
c_iv = c_1cmt_iv_bolus(t, dose=100, CL=5, Vz=30)
c_inf = c_1cmt_iv_infusion(t, dose=100, CL=5, Vz=30, t_inf=1.0)
c_oral = c_1cmt_oral(t, dose=100, CL_F=5, Vz_F=30, ka=0.8)

fig, axs = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
titles = ["IV Bolus", "IV Infusion (1h)", "Oral"]
for ax, c, title in zip(axs, [c_iv, c_inf, c_oral], titles, strict=False):
    ax.plot(t, c)
    ax.set_xlabel("Time (h)")
    ax.set_title(title)
axs[0].set_ylabel("Concentration (mg/L)")
plt.suptitle("1-Compartment Profiles")
plt.tight_layout()
plt.show()

C:\Users\priya\AppData\Local\Temp\ipykernel_6516\1179387837.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.2 Two-compartment models

In [26]:
from openpkflow.sim import c_2cmt_iv_infusion

t = np.linspace(0, 24, 300)

c_2iv = c_2cmt_iv_bolus(t, dose=100, CL=5, V1=10, Q=3, V2=40)
c_2inf = c_2cmt_iv_infusion(t, dose=100, CL=5, V1=10, Q=3, V2=40, t_inf=1.0)
c_2oral = c_2cmt_oral(t, dose=100, CL_F=5, V1_F=10, Q=3, V2=40, ka=0.8)

plt.figure(figsize=(8, 4))
plt.plot(t, c_2iv, label="2-cmt IV Bolus")
plt.plot(t, c_2inf, label="2-cmt IV Infusion")
plt.plot(t, c_2oral, label="2-cmt Oral")
plt.xlabel("Time (h)")
plt.ylabel("Concentration (mg/L)")
plt.title("Two-Compartment Profiles")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\priya\AppData\Local\Temp\ipykernel_6516\434430808.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.3 Superposition

In [27]:
# Superposition: sum independent dose contributions (linear PK only)
# superpose(times, dose_times, dose_amounts, unit_fn)
t = np.linspace(0, 36, 300)


def unit_oral(t_rel, dose):
    return c_1cmt_oral(t_rel, dose=dose, CL_F=5, Vz_F=30, ka=0.8)


c_combined = superpose(t, dose_times=[0, 12], dose_amounts=[100, 100], unit_fn=unit_oral)

# Individual contributions for illustration
c_dose1 = c_1cmt_oral(t, dose=100, CL_F=5, Vz_F=30, ka=0.8)
t2_mask = t > 12
c_dose2 = np.zeros_like(t)
c_dose2[t2_mask] = c_1cmt_oral(t[t2_mask] - 12, dose=100, CL_F=5, Vz_F=30, ka=0.8)

plt.figure(figsize=(8, 4))
plt.plot(t, c_dose1, "--", alpha=0.5, label="Dose 1 only")
plt.plot(t, c_dose2, "--", alpha=0.5, label="Dose 2 only")
plt.plot(t, c_combined, "k", linewidth=2, label="Combined (superpose)")
plt.axvline(12, color="gray", linestyle=":", label="2nd dose at t=12")
plt.xlabel("Time (h)")
plt.ylabel("Concentration (mg/L)")
plt.title("Superposition Example (q12h, 2 doses)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Cmax (combined) = {c_combined.max():.2f} mg/L")

ValueError: times must be strictly increasing.

### 3.4 High-level simulate() API with DoseRegimen

In [ ]:
from openpkflow.sim import Dose, DoseRegimen, OneCompartmentModel, TwoCompartmentModel, simulate

# --- Single oral dose ---
model = OneCompartmentModel(route="oral", CL_F=5.0, Vz_F=30.0, ka=0.8)
print(f"Model t1/2 = {model.half_life:.2f} h")

regimen = DoseRegimen(doses=(Dose(amount=100, time=0, route="oral"),))
sim_times = np.linspace(0, 24, 200)
result = simulate(model, regimen, sim_times, label="Single dose")
print(result.summary())

result.plot(show=True)

In [28]:
# --- Multiple oral doses (repeated dosing to steady-state) ---
md_regimen = DoseRegimen.from_repeated(amount=100, route="oral", tau=12, n_doses=7)
print("Dose times:", md_regimen.dose_times)

md_times = np.linspace(0, 84, 500)
md_result = simulate(model, md_regimen, md_times, label="Multiple dosing (7 doses, q12h)")
print(md_result.summary())

md_result.plot(show=True)

NameError: name 'DoseRegimen' is not defined

In [29]:
# --- Two-compartment IV infusion ---
model_2cmt = TwoCompartmentModel(route="iv_infusion", CL=8.0, V1=15.0, Q=4.0, V2=50.0)
iv_regimen = DoseRegimen(doses=(Dose(amount=200, time=0, route="iv_infusion", t_inf=0.5),))

t_iv = np.linspace(0, 24, 300)
iv_result = simulate(model_2cmt, iv_regimen, t_iv, label="2-cmt IV infusion")
print(iv_result.summary())

# Export results as dict
d = iv_result.to_dict()
print("to_dict keys:", list(d.keys()))

NameError: name 'TwoCompartmentModel' is not defined

---
## 4. IVIVC â€” In Vitro/In Vivo Correlation (Level A)

In [30]:
from openpkflow.ivivc import (
    convolution_predict,
    ivivc_predictability,
    levy_plot_data,
    loo_riegelman,
    wagner_nelson,
)
from openpkflow.ivivc.study import IVIVCStudy

# Synthetic data: oral and IV profiles
oral_times = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0]
oral_concs = [1.5, 3.2, 4.8, 6.0, 7.0, 7.0, 5.5, 3.8, 2.0]
iv_times = [0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0]
iv_concs = [10.0, 8.5, 6.5, 4.5, 3.2, 2.2, 1.0, 0.45, 0.10]
diss_times = [5.0, 10.0, 20.0, 30.0, 45.0, 60.0, 90.0, 120.0]
diss_pct = [5.0, 15.0, 35.0, 55.0, 75.0, 88.0, 97.0, 100.0]

### 4.1 Wagner-Nelson Deconvolution

In [31]:
# Option A: supply kel directly
fa_wn = wagner_nelson(oral_times, oral_concs, kel=0.12)

# Option B: estimate kel from IV UIR
fa_wn_uir = wagner_nelson(
    oral_times,
    oral_concs,
    iv_unit_impulse_times=iv_times,
    iv_unit_impulse_concs=iv_concs,
)

plt.figure(figsize=(7, 4))
plt.plot(oral_times, fa_wn, marker="o", label="Wagner-Nelson (kel given)")
plt.plot(oral_times, fa_wn_uir, marker="s", label="Wagner-Nelson (kel from UIR)")
plt.xlabel("Time (h)")
plt.ylabel("Cumulative Fraction Absorbed (F_a)")
plt.title("Wagner-Nelson Deconvolution")
plt.legend()
plt.tight_layout()
plt.show()

print("F_a at final time point (WN):", round(fa_wn[-1], 3))

F_a at final time point (WN): 1.0


C:\Users\priya\AppData\Local\Temp\ipykernel_6516\479358923.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.2 Loo-Riegelman Deconvolution (2-compartment)

In [32]:
fa_lr = loo_riegelman(
    oral_times,
    oral_concs,
    kel=0.18,
    k12=0.30,
    k21=0.40,
)
print("Loo-Riegelman F_a:", [round(v, 3) for v in fa_lr])
print(f"Terminal F_a = {fa_lr[-1]:.3f}")

Loo-Riegelman F_a: [np.float64(0.13), np.float64(0.326), np.float64(0.539), np.float64(0.737), np.float64(1.029), np.float64(1.216), np.float64(1.34), np.float64(1.301), np.float64(1.21)]
Terminal F_a = 1.210


### 4.3 Convolution Prediction

In [33]:
pred_times, pred_concs = convolution_predict(
    diss_times,
    diss_pct,
    iv_unit_impulse_times=iv_times,
    iv_unit_impulse_concs=iv_concs,
    dose_diss=100.0,
    dose_iv=100.0,
)

plt.figure(figsize=(7, 4))
plt.plot(pred_times, pred_concs, marker="o", label="Predicted in vivo")
plt.plot(oral_times, oral_concs, marker="s", linestyle="--", label="Observed in vivo")
plt.xlabel("Time (h)")
plt.ylabel("Concentration")
plt.title("IVIVC Convolution Prediction")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\priya\AppData\Local\Temp\ipykernel_6516\4066821977.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.4 Levy Plot and Predictability

In [34]:
# Levy plot correlates in vitro dissolution fraction to in vivo absorbed fraction
fa_norm = np.array(fa_wn)  # already as fractions (0-1)

levy_result = levy_plot_data(
    in_vitro_times=oral_times,
    in_vitro_fraction=np.array(diss_pct[: len(oral_times)]) / 100.0,
    in_vivo_fraction=fa_norm,
)
print("Levy plot:")
print(f"  Slope     = {levy_result['slope']:.3f}  (ideal: 1.0)")
print(f"  Intercept = {levy_result['intercept']:.3f}")
print(f"  R2        = {levy_result['r_squared']:.4f}")

# Visualise
plt.figure(figsize=(5, 4))
plt.scatter(levy_result["x"], levy_result["y"], color="steelblue")
x_line = np.linspace(0, 1, 100)
plt.plot(x_line, levy_result["slope"] * x_line + levy_result["intercept"], "r-", label="Fit")
plt.plot(x_line, x_line, "k--", alpha=0.4, label="1:1 line")
plt.xlabel("F dissolved (in vitro)")
plt.ylabel("F absorbed (in vivo)")
plt.title("Levy Plot")
plt.legend()
plt.tight_layout()
plt.show()

ValueError: in_vitro_fraction and in_vivo_fraction must be the same length

In [ ]:
# FDA 1997 predictability check: compare observed vs predicted Cmax and AUC
obs_cmax = 7.0
pred_cmax = float(np.max(pred_concs))
obs_auc = 55.0  # hypothetical AUCinf from the in vivo study
pred_auc = float(np.trapz(pred_concs, pred_times))

pred_result = ivivc_predictability(
    observed_cmax=obs_cmax,
    predicted_cmax=pred_cmax,
    observed_auc=obs_auc,
    predicted_auc=pred_auc,
)
print("Predictability:")
for k, v in pred_result.items():
    print(f"  {k}: {v}")

### 4.5 IVIVCStudy â€” integrated workflow

In [35]:
study = IVIVCStudy(
    in_vivo_times=oral_times,
    in_vivo_concs=oral_concs,
    dissolution_times=diss_times,
    dissolution_pct=diss_pct,
    iv_uir_times=iv_times,
    iv_uir_concs=iv_concs,
    method="wagner_nelson",
    kel=0.12,
    study_label="Prototype ER Tablet",
)
ivivc_result = study.analyze()

print(ivivc_result.summary())
print("to_dict keys:", list(ivivc_result.to_dict().keys()))

IVIVC Level A Analysis
Method: wagner_nelson
Study: Prototype ER Tablet

Levy Plot (IVIVC correlation)
---------------------------
Slope: -0.0773
Intercept: 1.0633
R-squared: 0.4733
N points (0.05-0.95): 8

Predictability Assessment (FDA 1997)
-------------------------------------
Cmax %PE: -92.10% (limit <= 15%)
AUCinf %PE: -87.76% (limit <= 15%)
Mean abs %PE: 89.93% (limit <= 10%)
Overall: FAIL

Disclaimer: This report was generated using OpenPKFlow -- an open-source Python workflow for pharmacometric analysis. Final regulatory interpretation should be reviewed by qualified formulation, pharmacokinetic, and regulatory experts.
to_dict keys: ['method', 'study_label', 'times', 'concentrations', 'fa', 'levy_plot_slope', 'levy_plot_intercept', 'levy_plot_r_squared', 'ivt_times', 'ivt_fraction', 'predicted_times', 'predicted_concs', 'predictability']


In [36]:
# Markdown report
md_content = ivivc_result.report("ivivc_report.md", format="markdown")
print(md_content[:600])

# IVIVC Level A Report: Prototype ER Tablet

**Generated:** 2026-05-22T11:31:52.116412+00:00
**OpenPKFlow version:** 2.0.0

## Study Parameters

| Parameter | Value |
|-----------|-------|
| Deconvolution method | wagner_nelson |
| In vivo time points | 9 |
| Dissolution time points | 8 |

## Levy Plot (IVIVC Correlation)

| Metric | Value |
|--------|------:|
| Slope | -0.0773 |
| Intercept | 1.0633 |
| R-squared | 0.4733 |
| N (0.05-0.95 range) | 8 |

## Predictability Assessment (FDA 1997)

| Metric | Value | Criterion | Status |
|--------|------:|-----------|--------|
| Cmax %PE | -92.10% 


---
## 5. Population PK â€” GOF and VPC

### 5.1 Goodness-of-Fit metrics

In [37]:
from openpkflow.pop import compute_iwres, obs_pred_metrics

# Simulated observed and predicted values
rng = np.random.default_rng(42)
dv = rng.lognormal(mean=1.5, sigma=0.2, size=50)
ipred = dv * rng.lognormal(mean=0, sigma=0.15, size=50)  # 15% proportional error

# IWRES
iwres = compute_iwres(dv, ipred, sigma=0.15)
print(f"IWRES: mean={iwres.mean():.3f}, std={iwres.std():.3f}")
print(f"IWRES in (-2, 2): {np.mean(np.abs(iwres) < 2) * 100:.1f}%")

# GOF metrics
metrics = obs_pred_metrics(dv, ipred)
print("\nGOF metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# Obs vs Pred scatter
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].scatter(ipred, dv, alpha=0.5)
lim = [min(dv.min(), ipred.min()) * 0.9, max(dv.max(), ipred.max()) * 1.1]
axs[0].plot(lim, lim, "r--", label="Identity")
axs[0].set_xlabel("IPRED")
axs[0].set_ylabel("DV")
axs[0].set_title("OBS vs PRED")
axs[0].legend()

axs[1].scatter(ipred, iwres, alpha=0.5)
axs[1].axhline(0, color="r", linestyle="--")
axs[1].axhline(2, color="gray", linestyle=":")
axs[1].axhline(-2, color="gray", linestyle=":")
axs[1].set_xlabel("IPRED")
axs[1].set_ylabel("IWRES")
axs[1].set_title("IWRES vs IPRED")
plt.tight_layout()
plt.show()

IWRES: mean=0.239, std=0.793
IWRES in (-2, 2): 100.0%

GOF metrics:
  n: 50.0000
  MPE: 0.1032
  RMSE: 0.5307
  rRMSE_pct: 11.4945
  R2: 0.4242


C:\Users\priya\AppData\Local\Temp\ipykernel_6516\3792625901.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.2 Visual Predictive Check (VPC)

In [38]:
from openpkflow.pop import simulate_vpc
from openpkflow.sim import Dose, DoseRegimen, OneCompartmentModel

# Build a synthetic observed dataset with TIME/DV columns (uppercase = default)
rng = np.random.default_rng(0)
t_per_subj = [0.5, 1.0, 2.0, 4.0, 8.0, 12.0, 24.0]
n_subj = 12
all_times, all_concs = [], []
for _ in range(n_subj):
    c = c_1cmt_oral(np.array(t_per_subj), dose=100, CL_F=5, Vz_F=30, ka=0.8)
    c = c * rng.lognormal(0, 0.25, size=len(t_per_subj))
    all_times.extend(t_per_subj)
    all_concs.extend(c.tolist())

obs_df = pd.DataFrame({"TIME": all_times, "DV": all_concs})

# simulate_vpc(model, regimen, observed_df, ...)
obs_model = OneCompartmentModel(route="oral", CL_F=5.0, Vz_F=30.0, ka=0.8)
vpc_regimen = DoseRegimen(doses=(Dose(100, 0, "oral"),))
vpc_result = simulate_vpc(
    obs_model,
    vpc_regimen,
    obs_df,
    n_replicates=500,
    n_bins=8,
    pi=(5.0, 50.0, 95.0),
    study_label="Theoph-like oral study",
)

print(vpc_result.summary())
vpc_result.plot(show=True)

ValueError: times must be strictly increasing.

---
## 6. Bayesian PK â€” MAP Estimation

In [ ]:
import math

from openpkflow.bayes import PKPrior, map_individual_pk

# Sparse clinical observation for a single subject (oral)
obs_times = [0.5, 1.0, 2.0, 4.0, 8.0, 12.0]
obs_concs = [1.8, 3.5, 5.2, 4.8, 3.4, 2.1]
dose_mg = 100.0

# Specify population priors (log-normal)
prior = PKPrior(
    log_cl_mean=math.log(5.0),  # pop mean CL_F = 5 L/h
    log_v_mean=math.log(30.0),  # pop mean Vz_F = 30 L
    log_ka_mean=math.log(0.8),  # pop mean ka = 0.8 1/h
    log_cl_sd=0.5,
    log_v_sd=0.5,
    log_ka_sd=0.5,
)

map_result = map_individual_pk(
    times=obs_times,
    concentrations=obs_concs,
    dose=dose_mg,
    route="oral",
    prior=prior,
    subject="SubjectA",
)

print(map_result.summary())
print(f"\nConverged: {map_result.converged}")
print(f"Gradient norm: {map_result.gradient_norm:.2e}")
print(f"Condition number: {map_result.condition_number:.1f}")
print(f"Uncertainty reliable: {map_result.uncertainty_reliable}")
if map_result.uncertainty_reliable:
    print(f"CL_F SE = {map_result.CL_F_se:.3f}")
    print(f"Vz_F SE = {map_result.Vz_F_se:.3f}")

In [39]:
# Plot observed vs MAP-fitted profile
t_plot = np.linspace(0, 24, 200)
c_map = c_1cmt_oral(
    t_plot,
    dose=dose_mg,
    CL_F=map_result.CL_F,
    Vz_F=map_result.Vz_F,
    ka=map_result.ka,
)

plt.figure(figsize=(7, 4))
plt.plot(t_plot, c_map, label="MAP-fitted profile")
plt.scatter(obs_times, obs_concs, color="red", zorder=5, label="Observed")
plt.xlabel("Time (h)")
plt.ylabel("Concentration (mg/L)")
plt.title("MAP Individual PK Estimation")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'dose_mg' is not defined

In [40]:
# MAP for IV bolus
iv_obs_times = [0.25, 1.0, 3.0, 8.0]
iv_obs_concs = [6.5, 4.8, 2.9, 1.1]

map_iv = map_individual_pk(
    times=iv_obs_times,
    concentrations=iv_obs_concs,
    dose=100.0,
    route="iv_bolus",
    subject="SubjectB",
)
print(f"IV MAP -- CL={map_iv.CL:.2f}, Vz={map_iv.Vz:.2f}, t1/2={map_iv.half_life:.2f} h")
print(f"Converged: {map_iv.converged}")

NameError: name 'map_individual_pk' is not defined

In [41]:
# MAP HTML report
map_result.report("map_pk_report.html", format="html")
print("MAP report written: map_pk_report.html")

NameError: name 'map_result' is not defined

### 6.1 Full Bayesian PK (requires PyMC) â€” optional

In [42]:
try:
    from openpkflow.bayes import bayes_individual_pk

    bayes_result = bayes_individual_pk(
        times=obs_times,
        concentrations=obs_concs,
        dose=dose_mg,
        route="oral",
        prior=prior,
        chains=2,
        draws=500,  # low draws for speed in demo
        subject="SubjectA",
    )
    print(bayes_result.summary())
    print(f"CL_F posterior: mean={bayes_result.cl_mean:.2f}, 95%CI={bayes_result.cl_95ci}")
    bayes_result.plot_posterior_kde(show=True)

except ImportError:
    print("PyMC not installed -- skipping full Bayesian PK.")
    print("Install with: pip install openpkflow[bayes]")

NameError: name 'obs_times' is not defined

### 6.2 Bayesian Bioequivalence (requires PyMC) â€” optional

In [43]:
try:
    from openpkflow.bayes import bayes_be

    rng_be = np.random.default_rng(7)
    ref_aucs = rng_be.lognormal(mean=np.log(100), sigma=0.2, size=24)
    tst_aucs = ref_aucs * rng_be.lognormal(mean=np.log(1.05), sigma=0.15, size=24)

    bayes_be_result = bayes_be(
        reference_values=ref_aucs,
        test_values=tst_aucs,
        chains=2,
        draws=500,
    )
    print(bayes_be_result.summary())

except ImportError:
    print("PyMC not installed -- skipping Bayesian BE.")
    print("Install with: pip install openpkflow[bayes]")

TypeError: bayes_be() got an unexpected keyword argument 'reference_values'

---
## 7. Bioequivalence (TOST)

In [44]:
from openpkflow.be import BEStudy, be_tost

# Simulate a 2x2 crossover study (24 subjects)
rng = np.random.default_rng(123)
n = 24
ref_auc = rng.lognormal(mean=np.log(100), sigma=0.25, size=n)
tst_auc = ref_auc * rng.lognormal(mean=np.log(1.03), sigma=0.20, size=n)

# Core TOST test (keyword args: be_lower, be_upper)
tost = be_tost(ref_auc, tst_auc, be_lower=0.80, be_upper=1.25)
print("TOST result:", tost)

TOST result: BETOSTResult(n=24, gmr=1.0553289718892311, gmr_lower_90ci=0.9865419739139967, gmr_upper_90ci=1.128912168318823, be_lower=0.8, be_upper=1.25, bioequivalent=True, cv_intra_pct=19.446544173767773, alpha=0.05)


In [45]:
# Full BEStudy with DataFrame input
be_df = pd.DataFrame(
    {
        "subject": np.arange(1, n + 1),
        "sequence": ["RT"] * (n // 2) + ["TR"] * (n // 2),
        "reference": ref_auc,
        "test": tst_auc,
    }
)

be_study = BEStudy(be_df, parameter="AUCinf")
be_result = be_study.analyze(be_lower=0.80, be_upper=1.25)

print(be_result.summary())
print(f"Bioequivalent: {be_result.bioequivalent}")
print(f"GMR (point estimate): {be_result.gmr:.4f}")
print(f"90% CI: [{be_result.gmr_lower_90ci:.4f}, {be_result.gmr_upper_90ci:.4f}]")
print(f"Intra-subject CV%: {be_result.cv_intra_pct:.1f}%")

Bioequivalence Summary
Parameter     : AUCinf
Subjects (n)  : 24
GMR (T/R)     : 1.0553
90% CI        : [0.9865, 1.1289]
Limits        : [0.8000, 1.2500]
CV (intra)    : 19.4%
Conclusion    : BIOEQUIVALENT


AttributeError: 'BEResult' object has no attribute 'conclusion'

In [46]:
# BEStudy.from_nca_results() wires NCA output -> BE in one line
print("BEStudy.from_nca_results() is available for wiring NCA -> BE in one line.")
print("Example:")
ref_arg = "ref_nca_results"
tst_arg = "tst_nca_results"
print(f"  be_study = BEStudy.from_nca_results({ref_arg}, {tst_arg}, parameter='AUCinf')")

# Generate BE HTML report
be_result.report("be_report.html", format="html")
print("BE report written: be_report.html")

BEStudy.from_nca_results() is available for wiring NCA -> BE in one line.
Example:
  be_study = BEStudy.from_nca_results(ref_nca_results, tst_nca_results, parameter='AUCinf')
BE report written: be_report.html


---
## 8. ML Surrogate â€” PKSurrogate (requires PyTorch)

In [47]:
try:
    from openpkflow.ml import PKSurrogate

    # Generate synthetic training data: [time, dose, CL_F, Vz_F, ka] -> concentration
    rng = np.random.default_rng(0)
    n_samples = 2000
    times_train = rng.uniform(0.1, 24, n_samples)
    doses_train = rng.uniform(50, 200, n_samples)
    cl_f_train = rng.uniform(3, 12, n_samples)
    vz_f_train = rng.uniform(20, 60, n_samples)
    ka_train = rng.uniform(0.3, 2.0, n_samples)

    X_train = np.column_stack([times_train, doses_train, cl_f_train, vz_f_train, ka_train])
    y_train = np.array(
        [
            c_1cmt_oral(np.array([t]), d, c, v, k)[0]
            for t, d, c, v, k in zip(
                times_train, doses_train, cl_f_train, vz_f_train, ka_train, strict=False
            )
        ]
    )

    surrogate = PKSurrogate(hidden_sizes=(64, 64), epochs=200, lr=1e-3, seed=42)
    surrogate.fit(X_train, y_train)

    # Test predictions
    t_test = np.linspace(0.1, 24, 50)
    X_test = np.column_stack(
        [
            t_test,
            np.full(50, 100),
            np.full(50, 5),
            np.full(50, 30),
            np.full(50, 0.8),
        ]
    )
    y_pred = surrogate.predict(X_test)
    y_true = c_1cmt_oral(t_test, 100, 5, 30, 0.8)

    plt.figure(figsize=(7, 4))
    plt.plot(t_test, y_true, "k-", label="Analytical (truth)")
    plt.plot(t_test, y_pred, "r--", label="MLP surrogate")
    plt.xlabel("Time (h)")
    plt.ylabel("Concentration (mg/L)")
    plt.title("PKSurrogate vs Analytical Model")
    plt.legend()
    plt.tight_layout()
    plt.show()

    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    print(f"Test RMSE: {rmse:.4f} mg/L")

except ImportError:
    print("PyTorch not installed -- skipping PKSurrogate.")
    print("Install with: pip install openpkflow[ml]")

PyTorch not installed -- skipping PKSurrogate.
Install with: pip install openpkflow[ml]


---
## 9. Report Generation

### 9.1 HTML reports (no extra deps)

In [48]:
# Dissolution comparison report
comparison.report("report_dissolution.html", format="html")
print("dissolution report -> report_dissolution.html")

# NCA summary report
results.report("report_nca_summary.html", format="html")
print("NCA summary report -> report_nca_summary.html")

# Single-subject NCA report
sub1 = results.results[0]
sub1.report("report_nca_sub1.html", format="html")
print("NCA single-subject report -> report_nca_sub1.html")

# IVIVC report
ivivc_result.report("report_ivivc.html", format="html")
print("IVIVC report -> report_ivivc.html")

# BE report
be_result.report("report_be.html", format="html")
print("BE report -> report_be.html")

dissolution report -> report_dissolution.html


NameError: name 'results' is not defined

### 9.2 PDF reports (requires reportlab)

In [ ]:
try:
    comparison.report("report_dissolution.pdf", format="pdf")
    print("PDF report -> report_dissolution.pdf")

    results.report("report_nca_summary.pdf", format="pdf")
    print("NCA PDF -> report_nca_summary.pdf")

    ivivc_result.report("report_ivivc.pdf", format="pdf")
    print("IVIVC PDF -> report_ivivc.pdf")

    be_result.report("report_be.pdf", format="pdf")
    print("BE PDF -> report_be.pdf")

    map_result.report("report_map.pdf", format="pdf")
    print("MAP PDF -> report_map.pdf")

except ImportError:
    print("reportlab not installed -- skipping PDF reports.")
    print("Install with: pip install openpkflow[reports]")

### 9.3 Word (.docx) reports (requires python-docx)

In [ ]:
try:
    comparison.report("report_dissolution.docx", format="docx")
    print("DOCX report -> report_dissolution.docx")

    results.report("report_nca_summary.docx", format="docx")
    print("NCA DOCX -> report_nca_summary.docx")

    ivivc_result.report("report_ivivc.docx", format="docx")
    print("IVIVC DOCX -> report_ivivc.docx")

    be_result.report("report_be.docx", format="docx")
    print("BE DOCX -> report_be.docx")

except ImportError:
    print("python-docx not installed -- skipping DOCX reports.")
    print("Install with: pip install openpkflow[reports]")

---
## 10. CLI Quick Reference

openpkflow ships a full CLI built with Typer. Run from a terminal:

In [ ]:
import subprocess

for cmd in [
    ["openpkflow", "version"],
    ["openpkflow", "similarity", "--reference", "20,40,60,80", "--test", "21,39,61,79"],
]:
    print("$", " ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout.strip())
    if out.stderr.strip():
        print("STDERR:", out.stderr.strip())
    print()

CLI commands available:
```
openpkflow version
openpkflow similarity --reference "20,40,60,80" --test "21,39,61,79"
openpkflow dissolution compare data.csv --reference reference --test test --report out.html
openpkflow be compare data.csv --parameter AUCinf --reference-col reference --test-col test
openpkflow ivivc run data.csv ...
```

---
## Summary

| Module | Key classes / functions |
|--------|------------------------|
| `dissolution` | `f1`, `f2`, `msd`, `max_deviation`, `bootstrap_f2`, `DissolutionStudy`, `fit_dissolution_models`, `model_dependent_comparison`, `MultiMediaStudy` |
| `nca` | `auc_linear/log/luld`, `cmax`, `tmax`, `lambda_z`, `auc_inf_obs`, `clearance_volume_parameters`, `steady_state_parameters`, `cumulative_urinary_excretion`, `NCAStudy`, `fit_sparse_1cmt_oral` |
| `sim` | `c_1cmt_*`, `c_2cmt_*`, `superpose`, `OneCompartmentModel`, `TwoCompartmentModel`, `DoseRegimen`, `simulate` |
| `ivivc` | `wagner_nelson`, `loo_riegelman`, `convolution_predict`, `levy_plot_data`, `ivivc_predictability`, `IVIVCStudy` |
| `pop` | `compute_iwres`, `obs_pred_metrics`, `simulate_vpc` |
| `bayes` | `PKPrior`, `map_individual_pk`, `bayes_individual_pk`*, `bayes_be`* |
| `be` | `be_tost`, `BEStudy` |
| `ml` | `PKSurrogate`* |
| Reports | `.report(path, format='html'/'pdf'/'docx')` on result objects |

\* Requires optional extras: `pip install openpkflow[bayes]` or `openpkflow[ml]`

> **Disclaimer:** This report was generated using OpenPKFlow (open-source). Final regulatory interpretation should be reviewed by qualified formulation, pharmacokinetic, and regulatory experts.